# Домашнее задание: MLflow — практика управления ML-экспериментами
В этом домашнем задании вы на практике познакомитесь с MLflow — инструментом для управления жизненным циклом моделей машинного обучения: от экспериментов и сравнения моделей до регистрации, версионирования и использования моделей в сервисах.
<br>**Инфраструктура и окружение**<br>
Все задания выполняются в подготовленном нами окружении на базе Docker Compose.
В docker-compose уже настроены:
  - MLflow Tracking Server
  - backend-хранилище для экспериментов (volume)
  - хранилище артефактов (S3)
  - Python-окружение с необходимыми библиотеками (mlflow, scikit-learn, pandas, numpy, fastapi и др.)

Вам не нужно ничего устанавливать локально, кроме:
  - Docker
  - Docker Compose

⚠️ Важные правила
  - Не меняйте docker-compose.yml без необходимости
  - Все эксперименты должны логироваться в MLflow (проверяется через UI)
  - Код должен быть воспроизводим: повторный запуск ноутбука не должен ломаться

**После ДЗ**
Пожалуйста, поделитесь вашими впечатлениями от ДЗ, если появится желание:

In [1]:
# Ваш комментарий к ДЗ здесь:

## Часть 1 (4 балла)

**Задание 1. (0.25б)** Настрой MLflow Tracking Server так, чтобы он смотрел на наш запущенный в той же docke-compose сети сервис mlflow-service 

In [2]:
import mlflow
mlflow.set_tracking_uri("http://mlflow-service:5000")

**Задание 2. (0.25б)** Создайте эксперимент с именем Surname_Name (подставьте ваши Имя и Фамилию). Обеспечьте проверку на уже созданный эксперимент, чтобы код не падал с ошибками, если эксперимент уже создан.

In [3]:
name = "Ovsiannikov_Daniil" 
if not mlflow.get_experiment_by_name(name):
    mlflow.create_experiment(name)

mlflow.set_experiment(name)

<Experiment: artifact_location='s3://ovsyannikov-test-bucket/mlflow/167651011856721165', creation_time=1766570630581, experiment_id='167651011856721165', last_update_time=1766570630581, lifecycle_stage='active', name='Ovsiannikov_Daniil', tags={}>

**Задание 3. (1б)** 
Используй любой датасет из sklearn (например, Diabetes или Wine).
Обучи простую модель (LinearRegression / LogisticRegression).
Заведите `mlflow.start_run()` и залогируйте вручную:
  - параметры (alpha, l1_ratio или любые гиперпараметры модели) **(0.25б)**
  - метрики (MSE / Accuracy) **(0.25б)**
  - артефакт: CSV-файл с предсказаниями **(0.25б)**
run_name="your_telegram_nickname"(укажите свой ник в тг) **(0.25б)**

In [4]:
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

data = load_wine()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

c_prm = 1.0
max_iter_prm = 5000

with mlflow.start_run(run_name='runftwoods'):
    model = LogisticRegression(C=c_prm, max_iter=max_iter_prm)
    model.fit(X_train,y_train)
    
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    
    mlflow.log_param('C', c_prm)
    mlflow.log_param('max_iter', max_iter_prm)
    mlflow.log_metric('accuracy', acc)
    
    df_pred = pd.DataFrame({'y_test':y_test, 'y_pred':y_pred})
    df_pred.to_csv('predictions.csv', index=False)
    mlflow.log_artifact('predictions.csv')
    
    print(acc)

2025/12/24 12:09:01 INFO mlflow.tracking._tracking_service.client: 🏃 View run runftwoods at: http://mlflow-service:5000/#/experiments/167651011856721165/runs/897ae12921de40a99b05f9c562423af7.
2025/12/24 12:09:01 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow-service:5000/#/experiments/167651011856721165.


1.0


**Задание 4. (1б)** Сравните несколько моделей через MLflow (GridSearch вручную).<br>
<br>Переберите минимум 4 набора гиперпараметров (например, Ridge с разными alpha). **(0.25б)**
<br>Для каждого варианта создай новый run. **(0.25б)**
<br>Добавьте тэги model_type=ridge и stage=research. **(0.25б)**
<br>В конце выведите таблицу: параметры — метрика — run_id. **(0.25б)**

In [5]:
c_prm = [0.001, 0.05, 1.0, 100.0]
result = []

for c_val in c_prm:
    with mlflow.start_run(run_name='runftwoods_Grid_Search'):
        
        model = LogisticRegression(C=c_val, max_iter=5000)
        model.fit(X_train, y_train)
        
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        
        mlflow.set_tag('model_type','log_red')
        mlflow.set_tag('stage','research')
        mlflow.log_param('C', c_val)
        mlflow.log_metric('accuracy', acc)
        
        run_id = mlflow.active_run().info.run_id
        result.append({
            "C": c_val, 
            "accuracy": acc, 
            "run_id": run_id
        })

result = pd.DataFrame(result)
display(result)

2025/12/24 12:09:02 INFO mlflow.tracking._tracking_service.client: 🏃 View run runftwoods_Grid_Search at: http://mlflow-service:5000/#/experiments/167651011856721165/runs/874638d4012c43149ac02957cb342193.
2025/12/24 12:09:02 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow-service:5000/#/experiments/167651011856721165.
2025/12/24 12:09:03 INFO mlflow.tracking._tracking_service.client: 🏃 View run runftwoods_Grid_Search at: http://mlflow-service:5000/#/experiments/167651011856721165/runs/ca63932d99544c03a679e13df79a75dc.
2025/12/24 12:09:03 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow-service:5000/#/experiments/167651011856721165.
2025/12/24 12:09:05 INFO mlflow.tracking._tracking_service.client: 🏃 View run runftwoods_Grid_Search at: http://mlflow-service:5000/#/experiments/167651011856721165/runs/71b6fac6401f4fa9bb68b6d5c7948636.
2025/12/24 12:09:05 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at:

,C,accuracy,run_id
0,0.001,0.888889,874638d4012c43149ac02957cb342193
1,0.050,1.000000,ca63932d99544c03a679e13df79a75dc
2,1.000,1.000000,71b6fac6401f4fa9bb68b6d5c7948636
3,100.000,1.000000,bbbb6850f1174d15a9f66f5de0621021


**Задание 5. (0.25б)** Используйте autologging.<br>
Включите `mlflow.sklearn.autolog()`.
Обучите любой sklearn-пайплайн (например, StandardScaler + RandomForest).
Убедись, что:
  - параметры RF(или другой модели на ваш выбор) залогированы автоматически
  - важность признаков сохранена как артефакт
  - модель записана в MLflow

In [41]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

mlflow.sklearn.autolog()

data = load_wine()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

with mlflow.start_run(run_name='runftwoods_autolog'):
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('rf', RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42))
    ])
    
    pipeline.fit(X_train, y_train)
    
    feature_importance = pipeline.named_steps['rf'].feature_importances_
    importance_df = pd.DataFrame({
        'feature': data.feature_names,
        'importance': feature_importance
    }).sort_values('importance', ascending=False)
    
    importance_df.to_csv("feature_importance.csv", index=False)
    mlflow.log_artifact("feature_importance.csv")

mlflow.sklearn.autolog(disable=True)

2025/12/24 13:00:09 INFO mlflow.tracking._tracking_service.client: 🏃 View run runftwoods_autolog at: http://mlflow-service:5000/#/experiments/167651011856721165/runs/42951a321cf944d6a9cf84c4652d50eb.
2025/12/24 13:00:09 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow-service:5000/#/experiments/167651011856721165.


**Задание 6. (0.5б)** Nested runs: логирование подэкспериментов.<br>
Постройте структуру:
```
main_run
 ├── model_lr
 ├── model_rf
 └── model_gb
```
Внутри одного main_run создай три дочерних run’а. Каждая модель — свой подэксперимент с параметрами и метриками. **(0.25б)**<br>
В main_run залогируй метрику: лучшая accuracy среди трёх. **(0.25б)**

In [45]:
import mlflow.sklearn

data = load_wine()

X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

models = {
    "model_lr": LogisticRegression(max_iter=5000),
    "model_rf": RandomForestClassifier(n_estimators=50),
    "model_gb": GradientBoostingClassifier(n_estimators=50),
}

scores = {}

with mlflow.start_run(run_name="main_run_runftwoods"):

    for name, model in models.items():
        with mlflow.start_run(run_name=name, nested=True):

            pipeline = Pipeline([
                ("scaler", StandardScaler()),
                ("model", model),
            ])

            pipeline.fit(X_train, y_train)
            predict = pipeline.predict(X_test)

            acc = accuracy_score(y_test, predict)
            scores[name] = acc

            mlflow.log_param("model_name", name)
            mlflow.log_metric("accuracy", acc)

            mlflow.sklearn.log_model(pipeline, artifact_path="model")

    best_accuracy = max(scores.values())
    mlflow.log_metric("best_accuracy", best_accuracy)

2025/12/24 13:03:31 WARNING mlflow.models.model: Input example should be provided to infer model signature if the model signature is not provided when logging the model.
2025/12/24 13:03:31 INFO mlflow.tracking._tracking_service.client: 🏃 View run model_lr at: http://mlflow-service:5000/#/experiments/167651011856721165/runs/fb3c9be750f6438d83d06a5684324771.
2025/12/24 13:03:31 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow-service:5000/#/experiments/167651011856721165.
2025/12/24 13:03:37 WARNING mlflow.models.model: Input example should be provided to infer model signature if the model signature is not provided when logging the model.
2025/12/24 13:03:37 INFO mlflow.tracking._tracking_service.client: 🏃 View run model_rf at: http://mlflow-service:5000/#/experiments/167651011856721165/runs/a2acd3f14fd34716a3fd936cb2790e7d.
2025/12/24 13:03:37 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow-service:5000/#/experiments/

**Задание 7. (0.5б)** MLflow Model Registry — регистрация и промоут модели.<br>
Выберите лучшую модель из предыдущих run’ов.<br>
Зарегистрируй её в Model Registry под именем best_wine_model (или по датасету).
Промоутите модель в стадию Staging. **(0.25)**<br>
Обновите описание модели — добавьте: **(0.25б)**
  - используемый датасет
  - дату
  - параметры

In [63]:
from mlflow.tracking import MlflowClient
from datetime import datetime

name = "Ovsiannikov_Daniil"
model_name = "best_wine_model" 

client = MlflowClient()

experiment = client.get_experiment_by_name(name)
experiment_id = experiment.experiment_id

runs = client.search_runs(
    experiment_ids=[experiment_id],
    filter_string="attributes.status = 'FINISHED'",
    order_by=["metrics.acc DESC"]
)

best_run = runs[0]
best_run_id = best_run.info.run_id

print('run_id:', best_run_id)

model_uri = f"runs:/{best_run_id}/model"

result = mlflow.register_model(
    model_uri=model_uri,
    name=model_name
)

model_version = result.version
print(f'Version: {model_version}')

client.transition_model_version_stage(
    name=model_name,
    version=model_version,
    stage="Staging",
    archive_existing_versions=True
)

dataset_name = "Wines dataset"
date = datetime.now().strftime("%Y-%m-%d")

params = best_run.data.params

description = f"""
Dataset: {dataset_name}
Date: {date}
Parameters:{params}
"""

client.update_model_version(
    name=model_name,
    version=model_version,
    description=description
)

Registered model 'best_wine_model' already exists. Creating a new version of this model...
2025/12/24 13:20:14 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: best_wine_model, version 8


run_id: ef004092fc5e4bc784704a876d0afd7f
Version: 8


Created version '8' of model 'best_wine_model'.
/tmp/ipykernel_13274/3035385835.py:33: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1766582414231, current_stage='Staging', description=('\n'
 'Dataset: Wines dataset\n'
 'Date: 2025-12-24\n'
 "Parameters:{'model_name': 'model_gb'}\n"), last_updated_timestamp=1766582414365, name='best_wine_model', run_id='ef004092fc5e4bc784704a876d0afd7f', run_link='', source='s3://ovsyannikov-test-bucket/mlflow/167651011856721165/ef004092fc5e4bc784704a876d0afd7f/artifacts/model', status='READY', status_message='', tags={}, user_id='', version='8'>

**Задание 8. (0.25б)**: Загрузка и инференс модели из Registry<br>
Через Python загрузить версию модели из Registry. **(0.5б)**
Сделайте предсказания на тестовом наборе. **(0.5б)**

In [64]:
import mlflow.pyfunc
import pandas as pd
from sklearn.metrics import accuracy_score

mlflow.set_tracking_uri("http://mlflow-service:5000")
MODEL_URI = "models:/best_wine_model/Staging"

model = mlflow.pyfunc.load_model(MODEL_URI)

df_X_test = pd.DataFrame(X_test, columns=data.feature_names)
predictions = model.predict(df_X_test)
    
acc = accuracy_score(y_test, predictions)
print(f"Результат: Accuracy = {acc:.4f}")

/opt/conda/lib/python3.11/site-packages/mlflow/store/artifact/utils/models.py:32: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest = client.get_latest_versions(name, None if stage is None else [stage])


Результат: Accuracy = 0.9444


/opt/conda/lib/python3.11/site-packages/sklearn/base.py:457: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


## Часть 2 (1 балл) - опциональная!

Возьмите наш DAG с MLFlow с Вебинара 6 (w6_ex4.py) и доработайте его так, чтобы вместо одного таска с обучением модели было 3 параллельных таска с обуением разных моделей. Если будете выполнять это задание - сдавайте ОТДЕЛЬНЫЙ .py-файл c DAG'ом (hw2.py). Ниже в строчке-комментарии напишите имя файла, если сдаете задание. Если не сдаете - оставьте строчку без изменений.

DAG должен содержать следующие шаги:
1. Инициализация пайплайна
   - лог времени старта
2. Сбор данных
   - загрузка CSV-датасета по URL
   - сохранение raw-данных в S3
3. Подготовка данных
   - очистка
   - one-hot encoding категориальных признаков
   - train/test split
   - сохранение X_train, X_test, y_train, y_test в S3
4. (4-6) Обучение моделей (НЕ ОДНА!)
  - минимум 3 модели
  - каждая модель обучается в отдельном Airflow-таске
  - все таски используют одну и ту же функцию обучения

Требования к обучению моделей:
 - Модели должны храниться в словаре MODELS
 - Выбор модели должен происходить по параметру model_name
 - Каждая модель:
   - логируется в MLflow
   - имеет отдельный run_name
   - регистрируется под уникальным именем

⚠️ Вы можете полностью переделать DAG под свой эксперимент, если у вас есть время и желание! (То есть вместо титаника обучать свой датасет и модели).

⚠️ Не хардкодьте бакеты и ключи доступа к S3, чтобы у проверяющих не ломался код. Можно просто оставить переменные как есть в примере с вебинара.

In [10]:
# Ваш ответ здесь: